# Trial Batch Runner — `base` Subtrials
Run every saved `TrialDefinition` once with the **base** ego configuration (no speed modifier) and record both ground-truth risk *and* predicted risk from the two model families:

- **E2E** (`QueuedE2ERiskInference` from a Seq2Seq E2E checkpoint) → `predicted_risk_e2e`
- **Composed BDU-GRU** (`QueuedComposedBDUGRURiskInference`, perception → 32-D → BDU-GRU) → `predicted_risk_composed`

For each trial:
1. Spawn the ego at the route start with `speed_multiplier=1.0`.
2. Drive the route at the base target speed, saving RGB frames + JSONL.
3. After the run, walk the JSONL and run **both** models on the saved frames, writing `predicted_risk_e2e` / `predicted_risk_composed` per record.

Output lands at `MIREIA/trials/<trial>/runs/<timestamp>_base/`. Companion notebooks:
- `trial_slow_batch_runner.ipynb` — same trials but with `speed_multiplier = SPEED_FACTOR` (default 0.75), ground-truth only, no model passes.
- *(later)* per-model trial runners that store ground truth + a single model's prediction.

## 0 — CARLA Launch
Skip if CARLA is already running.

In [1]:
import subprocess

AUTO_LAUNCH_CARLA = True
CARLA_EXE = "CarlaUE4.exe"

if AUTO_LAUNCH_CARLA:
    subprocess.Popen(CARLA_EXE, shell=True)
    print(f"Launched {CARLA_EXE}")
else:
    print("AUTO_LAUNCH_CARLA is False. Assuming CARLA is already running.")

Launched CarlaUE4.exe


## 1 — Imports

In [2]:
import json
import os
import sys
import time
from pathlib import Path

import torch
from ultralytics import YOLO

from MIREIA.config import Config
from MIREIA.data_collection.dataset_utils import load_jsonl_records, resolve_image_path
from MIREIA.perception import (
    DepthAnythingV2Estimator,
    FeatureIntegrator,
    QueuedComposedBDUGRURiskInference,
    QueuedE2ERiskInference,
    create_environment_classifier_predictor,
    load_road_segmentation_model,
)
from MIREIA.simulation.trials import EgoTrialConfig, TrialDefinition, TrialRunner

xFormers not available
xFormers not available


## 2 — Discover Trials
Lists every `trial.json` under `PATH_TO_TRIALS`. Use `INCLUDE_PREFIXES` / `EXCLUDE_TRIALS` to limit the run.

In [3]:
trials_root = Path(Config.PATH_TO_TRIALS)
all_trials = sorted(p.parent.name for p in trials_root.glob("*/trial.json") if p.is_file())

# Filtering (leave INCLUDE_PREFIXES empty to include every trial)
INCLUDE_PREFIXES: list[str] = []   # e.g. ["auto_"] to only run auto-generated trials
EXCLUDE_TRIALS:   list[str] = []# [trial for trial in all_trials if trial not in ["auto_20C_SoftRainSunset_Town10HD_HighVol"]]   # exact trial names to skip

if INCLUDE_PREFIXES:
    selected_trial_names = [n for n in all_trials if any(n.startswith(p) for p in INCLUDE_PREFIXES)]
else:
    selected_trial_names = list(all_trials)
selected_trial_names = [n for n in selected_trial_names if n not in EXCLUDE_TRIALS]

print(f"Found {len(all_trials)} trial(s); {len(selected_trial_names)} selected to run:")
for n in selected_trial_names:
    print(f"  - {n}")

Found 20 trial(s); 20 selected to run:
  - auto_17A_WetNoon_Town05_HighVol
  - auto_17B_WetNoon_Town05_LowVol
  - auto_17C_WetNoon_Town10HD_HighVol
  - auto_17D_WetNoon_Town10HD_LowVol
  - auto_18A_MidRainyNoon_Town05_HighVol
  - auto_18B_MidRainyNoon_Town05_LowVol
  - auto_18C_MidRainyNoon_Town10HD_HighVol
  - auto_18D_MidRainyNoon_Town10HD_LowVol
  - auto_19A_CloudySunset_Town05_HighVol
  - auto_19B_CloudySunset_Town05_LowVol
  - auto_19C_CloudySunset_Town10HD_HighVol
  - auto_19D_CloudySunset_Town10HD_LowVol
  - auto_20A_SoftRainSunset_Town05_HighVol
  - auto_20B_SoftRainSunset_Town05_LowVol
  - auto_20C_SoftRainSunset_Town10HD_HighVol
  - auto_20D_SoftRainSunset_Town10HD_LowVol
  - auto_21A_ClearNoon_Town05_HighVol_NoFog_Night
  - auto_21B_CloudyNoon_Town05_LowVol_NoFog_Night
  - auto_21C_WetNoon_Town10HD_HighVol_NoFog_Night
  - auto_21D_HardRainNoon_Town10HD_LowVol_NoFog_Night


## 3 — Base Subtrial Config
Single ego configuration reused across every trial: stock blueprint, base target speed, behavior agent, **no speed modifier** (`speed_multiplier=1.0`). Subtrial folder name is `base`, so each run lands at `runs/<timestamp>_base/`.

In [4]:
MAX_STEPS_PER_TRIAL = 6000   # hard cap so one stuck trial cannot block the batch

base_ego_cfg = EgoTrialConfig(
    name="base",
    ego_blueprint="vehicle.lincoln.mkz_2020",
    #target_speed_kmh=20.0,
    speed_multiplier=1.0,                # no speed function applied -> "base" run
    use_vehicle_camera_defaults=True,
    controller_mode="behavior_agent",
    controller_behavior="normal",
)

runner = TrialRunner(verbose=False)
print("Base ego config ready.")

Base ego config ready.


## 4 — Load Inference Models
Loads `QueuedE2ERiskInference` and `QueuedComposedBDUGRURiskInference` once and reuses them across every trial. The composed predictor needs a YOLO + depth checkpoint; environment classifier and road segmentation are optional and skipped if their checkpoints are missing. Whichever predictor's checkpoint is missing is set to `None`, and the corresponding `predicted_risk_*` field is simply not written for that trial.

In [5]:
device_name = "cuda" if torch.cuda.is_available() else "cpu"

E2E_CHECKPOINT      = Path(Config.PATH_TO_MODELS) / "e2e_risk_checkpoint.pt"
COMPOSED_CHECKPOINT = Path(Config.PATH_TO_MODELS) / "bdu_gru_search_02.pt"
YOLO_CHECKPOINT     = Path(Config.PATH_TO_MODELS) / "yolo11s.pt"
DEPTH_CHECKPOINT    = Path(Config.PATH_TO_MODELS) / "depth_anything_v2_vits.pth"
CLIMATE_CHECKPOINT  = Path(Config.PATH_TO_MODELS) / "environment_multitask_checkpoint.pt"
ROAD_CHECKPOINT     = Path(Config.PATH_TO_MODELS) / "road_segmentation_multitask_checkpoint.pt"

print(f"Device: {device_name}")

# ---- E2E predictor ----------------------------------------------------------
e2e_predictor = None
if E2E_CHECKPOINT.is_file():
    e2e_predictor = QueuedE2ERiskInference.from_checkpoint(
        checkpoint_path=str(E2E_CHECKPOINT),
        device=device_name,
    )
    print(f"E2E loaded: {E2E_CHECKPOINT.name}")
else:
    print(f"[skip] E2E checkpoint missing: {E2E_CHECKPOINT}")

# ---- Composed BDU-GRU predictor --------------------------------------------
composed_predictor = None
required_for_composed = {
    "composed BDU-GRU": COMPOSED_CHECKPOINT,
    "YOLO":             YOLO_CHECKPOINT,
    "depth":            DEPTH_CHECKPOINT,
}
missing_required = [name for name, p in required_for_composed.items() if not p.is_file()]
if missing_required:
    print(f"[skip] Composed predictor missing: {', '.join(missing_required)}")
else:
    yolo_model = YOLO(str(YOLO_CHECKPOINT))
    depth_estimator = DepthAnythingV2Estimator(
        checkpoint_path=DEPTH_CHECKPOINT,
        encoder="vits",
        device=device_name,
    )
    environment_predictor = (
        create_environment_classifier_predictor(checkpoint_path=str(CLIMATE_CHECKPOINT), device=device_name)
        if CLIMATE_CHECKPOINT.is_file() else None
    )
    road_segmentation = (
        load_road_segmentation_model(checkpoint_path=str(ROAD_CHECKPOINT), device=device_name)
        if ROAD_CHECKPOINT.is_file() else None
    )

    composed_predictor = QueuedComposedBDUGRURiskInference.from_checkpoint(
        checkpoint_path=str(COMPOSED_CHECKPOINT),
        feature_integrator=FeatureIntegrator(),
        yolo_model=yolo_model,
        depth_estimator=depth_estimator,
        environment_predictor=environment_predictor,
        road_segmentation=road_segmentation,
        device=device_name,
    )
    print(f"Composed loaded: {COMPOSED_CHECKPOINT.name}")
    print(f"  environment_predictor: {environment_predictor is not None}")
    print(f"  road_segmentation:     {road_segmentation is not None}")

if e2e_predictor is None and composed_predictor is None:
    print("\n[warn] Both predictors unavailable - JSONLs will only contain ground truth.")

Device: cuda
E2E loaded: e2e_risk_checkpoint.pt
Composed loaded: bdu_gru_search_02.pt
  environment_predictor: True
  road_segmentation:     True


### Inference Helper
After each trial finishes, this helper walks the run's `dataset.jsonl`, feeds every RGB frame through both predictors (`reset_queue()` per trial so the temporal buffer is clean), and writes `predicted_risk_e2e` / `predicted_risk_composed` back into each record.

In [6]:
def predict_and_annotate_jsonl(jsonl_path: Path,
                               e2e_predictor,
                               composed_predictor) -> dict:
    """Walk dataset.jsonl, run both predictors on each RGB frame, and write
    `predicted_risk_e2e` / `predicted_risk_composed` per record.

    Predictors that are None are skipped. Records with missing/unreadable images
    keep their existing fields. Returns a small stats dict for logging.
    """
    if e2e_predictor is None and composed_predictor is None:
        return {"frames": 0, "valid_frames": 0, "e2e_count": 0, "composed_count": 0}

    records = load_jsonl_records(str(jsonl_path))
    if not records:
        return {"frames": 0, "valid_frames": 0, "e2e_count": 0, "composed_count": 0}

    jsonl_dir = jsonl_path.parent
    paths: list[str | None] = []
    for rec in records:
        rel = str(rec.get("rgb_image_path", "")).strip()
        if not rel:
            paths.append(None)
            continue
        full = resolve_image_path(str(jsonl_dir), rel, normalize_paths=True)
        paths.append(full if full and os.path.isfile(full) else None)

    valid_count = sum(1 for p in paths if p)
    if valid_count == 0:
        return {"frames": len(records), "valid_frames": 0, "e2e_count": 0, "composed_count": 0}

    def _run(predictor) -> list:
        if predictor is None:
            return [None] * len(paths)
        predictor.reset_queue()
        out_list = []
        for p in paths:
            if p is None:
                out_list.append(None)
                continue
            out = predictor.add_image_path(p)
            val = float(out.latest_risk) if out.ready and out.latest_risk is not None else None
            out_list.append(val)
        return out_list

    e2e_preds = _run(e2e_predictor)
    composed_preds = _run(composed_predictor)

    for rec, e2e_v, comp_v in zip(records, e2e_preds, composed_preds):
        if e2e_v is not None:
            rec["predicted_risk_e2e"] = e2e_v
        if comp_v is not None:
            rec["predicted_risk_composed"] = comp_v

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    return {
        "frames":         len(records),
        "valid_frames":   valid_count,
        "e2e_count":      sum(1 for v in e2e_preds if v is not None),
        "composed_count": sum(1 for v in composed_preds if v is not None),
    }


print("Helper ready: predict_and_annotate_jsonl()")

Helper ready: predict_and_annotate_jsonl()


## 5 — Run Each Trial
For each trial: spawn the ego, drive the route saving RGB frames + JSONL, then run **E2E** and **Composed BDU-GRU** predictors over the saved frames and annotate every record. Failures don't abort the batch — they're recorded and reported at the end.

`store_topdown_images=False` and `draw_debug_every_tick=False` keep things lean. RGB frames are the only images written; you can purge them later if disk is tight.

In [7]:
all_summaries = []
inference_stats: dict[str, dict] = {}
failed: list[tuple[str, str]] = []

# --- Run-time accounting (sim vs wall, cap meaning) ----------------------------
# step  = one CARLA world.tick() = one fixed_delta seconds of simulated time.
# Wall time per tick depends on CPU/GPU. With RGB images saved (needed by the
# post-run predictors) you'll see lower speedup than the JSONL-only fast path.
# image_stride controls JSONL + RGB cadence, NOT loop iteration count.
# The loop exits early when controller.done() — ego reached the route end
# (within arrival_threshold_m = 3 m). max_steps is the cap, applied only if the
# ego gets stuck or the route is longer than the cap.
fixed_delta_s    = float(Config.SIM_FIXED_DELTA_SECONDS)   # most trials use this default
stride_ticks     = int(Config.RECORD_EVERY_N_TICKS)
sim_time_cap_s   = MAX_STEPS_PER_TRIAL * fixed_delta_s
max_jsonl_frames = MAX_STEPS_PER_TRIAL // stride_ticks
record_hz        = 1.0 / (fixed_delta_s * stride_ticks)

print("Run configuration:")
print(f"  fixed_delta:          {fixed_delta_s:.3f} s/tick  ({1/fixed_delta_s:.1f} Hz physics)")
print(f"  image_stride:         {stride_ticks} ticks/frame  ({record_hz:.2f} Hz JSONL records + RGB)")
print(f"  max_steps_per_trial:  {MAX_STEPS_PER_TRIAL}  (cap = {sim_time_cap_s:.1f} s sim time, "
      f"<= {max_jsonl_frames} JSONL frames)")
print(f"  finished=True   -> controller reported done before the cap (good)")
print(f"  finished=False  -> hit max_steps cap (stuck or route longer than cap; raise MAX_STEPS_PER_TRIAL)")
print(f"  duration_s in summary is WALL-clock seconds, not sim seconds.")
print(f"  num_frames in summary is JSONL record count, not tick count.")
print(f"  E2E predictor:      {'ON' if e2e_predictor      is not None else 'off'}")
print(f"  Composed predictor: {'ON' if composed_predictor is not None else 'off'}")

# Mutable state shared with the progress callback
_progress_state: dict = {"trial_started": 0.0}

def _format_secs(s: float) -> str:
    if s < 60:
        return f"{s:5.1f}s"
    m, sec = divmod(s, 60)
    return f"{int(m):02d}:{int(sec):02d}"

def _on_progress(step: int, max_steps: int) -> None:
    """Live progress print (carriage-return updated). Throttled to ~5 Hz wall."""
    now = time.time()
    last = _progress_state.get("last_print", 0.0)
    if now - last < 0.2:
        return
    _progress_state["last_print"] = now
    elapsed = now - _progress_state["trial_started"]
    rate = step / elapsed if elapsed > 0 else 0.0
    pct = 100.0 * step / max_steps if max_steps else 0.0
    eta_to_cap = (max_steps - step) / rate if rate > 0 else float("inf")
    speedup = (step * fixed_delta_s) / elapsed if elapsed > 0 else 0.0
    sys.stdout.write(
        f"\r    step {step:5d}/{max_steps} ({pct:5.1f}% of cap)  "
        f"wall={_format_secs(elapsed)}  "
        f"rate={rate:5.1f} t/s  "
        f"speedup={speedup:4.2f}x  "
        f"eta_to_cap={_format_secs(eta_to_cap) if rate > 0 else '  inf'}    "
    )
    sys.stdout.flush()

batch_started = time.time()
for idx, trial_name in enumerate(selected_trial_names, start=1):
    print()
    print(f">>> [{idx}/{len(selected_trial_names)}] {trial_name}", flush=True)
    _progress_state["trial_started"] = time.time()
    _progress_state["last_print"] = 0.0
    try:
        trial = TrialDefinition.load(trial_name)
        summary = runner.run_subtrial(
            trial=trial,
            ego_cfg=base_ego_cfg,
            max_steps=MAX_STEPS_PER_TRIAL,
            image_stride=Config.RECORD_EVERY_N_TICKS,
            store_rgb_images=True,           # need RGB on disk for post-run inference
            store_topdown_images=False,
            store_risk_frame_images=False,
            store_static_risk_map=False,
            draw_debug_every_tick=False,
            progress_callback=_on_progress,
            progress_every_n_steps=25,
        )
        elapsed = time.time() - _progress_state["trial_started"]
        sim_seconds = summary.num_frames * fixed_delta_s * stride_ticks
        speedup = sim_seconds / elapsed if elapsed > 0 else float("inf")
        # Clear the live-progress line before printing the final result.
        sys.stdout.write("\r" + " " * 130 + "\r")
        sys.stdout.flush()
        print(
            f"    sim ok in {elapsed:6.1f}s wall  "
            f"frames={summary.num_frames:4d}  "
            f"sim_t={sim_seconds:6.1f}s  "
            f"speedup={speedup:5.2f}x  "
            f"dist={summary.traveled_m:7.1f}m  "
            f"finished={summary.finished}"
        )

        # --- Post-run inference: annotate dataset.jsonl with both predictions ----
        jsonl_path = Path(summary.run_path) / "dataset.jsonl"
        if jsonl_path.is_file() and (e2e_predictor is not None or composed_predictor is not None):
            t_inf = time.time()
            stats = predict_and_annotate_jsonl(jsonl_path, e2e_predictor, composed_predictor)
            inf_elapsed = time.time() - t_inf
            inference_stats[trial_name] = stats
            print(
                f"    inference in {inf_elapsed:6.1f}s wall  "
                f"valid_frames={stats['valid_frames']}/{stats['frames']}  "
                f"e2e={stats['e2e_count']}  composed={stats['composed_count']}"
            )
        else:
            inference_stats[trial_name] = {"frames": summary.num_frames, "valid_frames": 0,
                                           "e2e_count": 0, "composed_count": 0}
            print("    inference skipped (no predictors loaded or JSONL missing)")

        all_summaries.append(summary)
    except Exception as e:
        elapsed = time.time() - _progress_state["trial_started"]
        sys.stdout.write("\r" + " " * 130 + "\r")
        sys.stdout.flush()
        print(f"    FAILED in {elapsed:.1f}s: {type(e).__name__}: {e}")
        failed.append((trial_name, f"{type(e).__name__}: {e}"))

batch_elapsed = time.time() - batch_started
print()
print(f"Batch done in {batch_elapsed:.1f}s (ok={len(all_summaries)}, failed={len(failed)})")

Run configuration:
  fixed_delta:          0.050 s/tick  (20.0 Hz physics)
  image_stride:         5 ticks/frame  (4.00 Hz JSONL records + RGB)
  max_steps_per_trial:  6000  (cap = 300.0 s sim time, <= 1200 JSONL frames)
  finished=True   -> controller reported done before the cap (good)
  finished=False  -> hit max_steps cap (stuck or route longer than cap; raise MAX_STEPS_PER_TRIAL)
  duration_s in summary is WALL-clock seconds, not sim seconds.
  num_frames in summary is JSONL record count, not tick count.
  E2E predictor:      ON
  Composed predictor: ON

>>> [1/20] auto_17A_WetNoon_Town05_HighVol
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=False, tm_lights=False)
Spawned 80 / 80 requested vehicles.
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed bec

KeyboardInterrupt: 

## 6 — Summary

In [8]:
if not all_summaries and not failed:
    print("No trials were run.")
else:
    print(f"=== Successful runs ({len(all_summaries)}) ===")
    print(
        f"  {'trial_name':50s} {'frames':>6s} {'dist_m':>9s} "
        f"{'risk_auc':>9s} {'risk/m':>9s} {'e2e':>5s} {'comp':>5s}  finished  run_path"
    )
    for s in all_summaries:
        stats = inference_stats.get(s.trial_name, {})
        e2e_c = stats.get("e2e_count", 0)
        cmp_c = stats.get("composed_count", 0)
        print(
            f"  {s.trial_name:50s} {s.num_frames:6d} {s.traveled_m:9.1f} "
            f"{s.risk_auc:9.3f} {s.risk_per_meter:9.5f} {e2e_c:5d} {cmp_c:5d}  "
            f"{str(s.finished):>8s}  {s.run_path}"
        )

    if failed:
        print()
        print(f"=== Failed runs ({len(failed)}) ===")
        for name, err in failed:
            print(f"  {name}: {err}")

=== Successful runs (6) ===
  trial_name                                         frames    dist_m  risk_auc    risk/m   e2e  comp  finished  run_path
  auto_17A_WetNoon_Town05_HighVol                       395     548.9   240.853   2.71390   380   380      True  t:\TFG\MIREIA\trials\auto_17A_WetNoon_Town05_HighVol\runs\20260510_233143_base
  auto_17B_WetNoon_Town05_LowVol                        141     559.3    64.644   2.03109   126   126      True  t:\TFG\MIREIA\trials\auto_17B_WetNoon_Town05_LowVol\runs\20260510_233523_base
  auto_17C_WetNoon_Town10HD_HighVol                     250     160.0   146.644   3.68901   235   235      True  t:\TFG\MIREIA\trials\auto_17C_WetNoon_Town10HD_HighVol\runs\20260510_233631_base
  auto_17D_WetNoon_Town10HD_LowVol                      640     422.6   371.348   3.65523   625   625      True  t:\TFG\MIREIA\trials\auto_17D_WetNoon_Town10HD_LowVol\runs\20260510_233856_base
  auto_18A_MidRainyNoon_Town05_HighVol                 1120     949.2  1157.736 